# ChaoSSM — Phase 3b': Harness Fix to Ceiling (+ GRU control)

## Why this phase

Every arm tested so far (Phases 1-3a) tops out around 0.06-0.23 accuracy on 12-token copy,
against a chance floor of 0.0625. A referee will not read a small gap between two failing
models as an architecture finding -- they will read it as a broken or undertrained harness.
This phase does not compare architectures yet (that is Phase 4). It asks a narrower, prior
question: **can this task be solved at all, by anything, in this harness?**

Changes from every prior phase:

1. **Depth + residual/MLP block.** Each layer is now `x -> LN -> recurrent cell -> residual
   add -> LN -> MLP(4x, GELU) -> residual add`, matching the block structure used by modern
   SSM/RNN papers (Mamba, S4, etc.) instead of a single bare recurrent step. Two such blocks
   are stacked.
2. **Longer training with a real schedule.** 5000 steps (up from 1200), 5% linear warmup,
   cosine decay to 0.1x peak lr, instead of a flat constant lr.
3. **A GRU arm** (`nn.GRUCell`) is added as the trainability reference. GRUs solve 12-token
   copy essentially perfectly within a few thousand steps on any reasonable harness; if GRU
   does not reach ceiling here either, the problem is the harness (task encoding, evaluation,
   optimizer), not any architecture's inductive bias.
4. **A fixed, held-out evaluation set** (2000 sequences, fixed seed 12345, generated once and
   reused for every arm/seed/checkpoint) replaces last-training-batch accuracy. Evaluated every
   250 steps and at the end.
5. **A proper stats module**: Cliff's delta (effect size), bootstrap 95% CI on the mean
   difference (10,000 resamples), and Holm-Bonferroni correction across all arm-pair
   comparisons on this task -- resolving the floor-p-value-only reporting used through Phase 3a.
6. **An automatic fallback lr sweep** for any arm that does not clear the ceiling bar (0.9) at
   its default lr: a short 3-seed, half-budget sweep over a small lr grid, with the best lr
   then used for that arm's full 8-seed, full-budget final run. This is the "3-point sweep"
   called for by pre-registered rule H-A, run automatically rather than left as a manual
   follow-up.

## Task and scope

Copy task only (L=12, vocab=16), the same task used in every prior phase, so the *only*
things that changed are the ones listed above. Selective copy and recall return in the full
battery (Phase 3b), once the ceiling question here is settled.

## Decision rule (H-A, pre-registered)

> On this harness, if GRU reaches >= 0.9 mean held-out accuracy and at least one chaotic arm
> also reaches >= 0.9, both after the automatic fallback sweep if needed, the ceiling question
> is closed and Phase 4 proceeds as planned. If GRU reaches >= 0.9 but no chaotic arm does, that
> is itself reported as a limitation ("chaos impairs trainability on copy at this scale"),
> Phase 4's periodic-r ablation is run anyway (it still answers a different question: is
> whatever chaos *does* achieve attributable to chaos or to nonlinearity), and the manuscript
   framing shifts accordingly. If GRU itself does not reach >= 0.9, the harness still has an
   unresolved problem and no architecture claims can be made until it is found.

## Setup

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase3b_prime"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase3b_prime


## Configuration

In [ ]:
SMOKE_TEST = False   # full run: 5 arms x 8 seeds x 5000 steps, copy task, held-out eval

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    n_layers: int = 2
    mlp_mult: int = 4
    steps: int = 5000
    batch: int = 64
    warmup_frac: float = 0.05
    lr_min_frac: float = 0.1          # cosine decays to lr_min_frac * peak lr
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    copy_len: int = 12
    arms: tuple = ("linear", "gru", "mamba2", "chaossm_free", "chaossm_corridor")
    seeds: int = 8
    eval_n: int = 2000
    eval_seed: int = 12345
    eval_every: int = 250
    ceiling_bar: float = 0.9
    # default (pre-chosen, documented) lr per arm -- NOT tuned on this task's results;
    # linear/chaossm defaults carried over from Phase 1-3; gru/mamba2 use standard-practice
    # defaults for their respective cell types (mamba2's matches the Phase 3a-ii correction).
    default_lr: dict = field(default_factory=lambda: dict(
        linear=3e-4, chaossm_free=3e-4, chaossm_corridor=3e-4, mamba2=1e-3, gru=1e-3))
    sweep_grid: tuple = (1e-4, 3e-4, 1e-3, 3e-3)
    sweep_seeds: int = 3
    sweep_step_frac: float = 0.5      # half budget for the sweep

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 120; CFG.seeds = 2; CFG.eval_n = 60; CFG.eval_every = 30
    CFG.sweep_seeds = 2; CFG.sweep_step_frac = 0.5
    print("*** SMOKE TEST -- pipeline check only ***")
print(json.dumps({k: (v if not isinstance(v, dict) else v) for k, v in asdict(CFG).items()}, indent=2, default=str))

{
  "state_dim": 48,
  "d_model": 48,
  "vocab": 16,
  "n_layers": 2,
  "mlp_mult": 4,
  "steps": 5000,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "gamma_init": 1.0,
  "dual_lr": 0.05,
  "alpha_max": 50.0,
  "corridor_target": 0.15,
  "corridor_half_width": 0.1,
  "lyap_window": 4,
  "lyap_subsample": 3,
  "lyap_every": 20,
  "copy_len": 12,
  "arms": [
    "linear",
    "gru",
    "mamba2",
    "chaossm_free",
    "chaossm_corridor"
  ],
  "seeds": 8,
  "eval_n": 2000,
  "eval_seed": 12345,
  "eval_every": 250,
  "ceiling_bar": 0.9,
  "default_lr": {
    "linear": 0.0003,
    "chaossm_free": 0.0003,
    "chaossm_corridor": 0.0003,
    "mamba2": 0.001,
    "gru": 0.001
  },
  "sweep_grid": [
    0.0001,
    0.0003,
    0.001,
    0.003
  ],
  "sweep_seeds": 3,
  "sweep_step_frac": 0.5
}


## 1. Cells (arms) -- unchanged mechanisms from Phase 3a, now used inside a block

In [ ]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    '''Shared wrapper for linear/chaossm arms: gate + fixed A/B + phi(.). Unchanged from
    Phase 3a except d_model/state_dim are always equal in this phase (both = state_dim).'''
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.state_dim) / math.sqrt(cfg.state_dim))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.state_dim) / math.sqrt(cfg.state_dim))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev, dtype):
        return torch.zeros(b, self.N, device=dev, dtype=dtype) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev, dtype=dtype)


class Mamba2Cell(nn.Module):
    '''Matched-simplification selective-SSM baseline (Mamba-2 style). Unchanged mechanism
    from Phase 3a: Delta_t and B_t are functions of the current input; A is a fixed,
    learnable per-channel decay rate.'''
    def __init__(self, cfg):
        super().__init__()
        self.N = cfg.state_dim
        A_init = torch.arange(1, cfg.state_dim + 1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A_init))
        self.W_delta = nn.Linear(cfg.state_dim, cfg.state_dim)
        self.W_B = nn.Linear(cfg.state_dim, cfg.state_dim)
        nn.init.zeros_(self.W_delta.bias); nn.init.zeros_(self.W_B.bias)
    def step(self, h, x):
        delta = F.softplus(self.W_delta(x))
        A = -torch.exp(self.A_log)
        A_bar = torch.exp(delta * A)
        B_t = self.W_B(x)
        return A_bar * h + delta * B_t
    def init_state(self, b, dev, dtype):
        return torch.zeros(b, self.N, device=dev, dtype=dtype)


class GRUArmCell(nn.Module):
    '''Trainability reference. Standard nn.GRUCell, same state_dim as every other arm.'''
    def __init__(self, cfg):
        super().__init__()
        self.N = cfg.state_dim
        self.cell = nn.GRUCell(cfg.state_dim, cfg.state_dim)
    def step(self, h, x):
        return self.cell(x, h)
    def init_state(self, b, dev, dtype):
        return torch.zeros(b, self.N, device=dev, dtype=dtype)


def make_cell(cfg, arm):
    if arm == "mamba2": return Mamba2Cell(cfg)
    if arm == "gru": return GRUArmCell(cfg)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    return ChaoSSMCell(cfg, phi_kind)

NEEDS_TRAJ = ("chaossm_free", "chaossm_corridor")   # arms whose corridor/Lyapunov diagnostics apply

## 2. Block: LN -> cell -> residual -> LN -> MLP -> residual

This is the addition that separates Phase 3b' from every prior phase. Each layer wraps its
recurrent cell in the pre-norm residual + feed-forward pattern used by every modern
sequence-model block (Transformer, Mamba, S4-block), instead of the bare single-step cell
used through Phase 3a. `d_model == state_dim` throughout, so the residual add needs no
projection.

In [ ]:
class Block(nn.Module):
    def __init__(self, cfg, arm):
        super().__init__()
        self.arm = arm
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.cell = make_cell(cfg, arm)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        hidden = cfg.d_model * cfg.mlp_mult
        self.mlp = nn.Sequential(
            nn.Linear(cfg.d_model, hidden), nn.GELU(), nn.Linear(hidden, cfg.d_model))
    def step(self, h, x):
        u = self.ln1(x)
        h = self.cell.step(h, u)
        y = x + h
        y = y + self.mlp(self.ln2(y))
        return h, y
    def init_state(self, b, dev, dtype):
        return self.cell.init_state(b, dev, dtype)


class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, arm, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.arm = arm
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, arm) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.readout = nn.Linear(cfg.d_model, readout_size)

    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok)
        dtype = x.dtype
        states = [blk.init_state(B, tok.device, dtype) for blk in self.blocks]
        outs = []
        trajs = [[] for _ in self.blocks] if return_traj else None
        xseqs = [[] for _ in self.blocks] if return_traj else None
        xt_seq = x
        for t in range(T):
            xt = xt_seq[:, t]
            for li, blk in enumerate(self.blocks):
                h_in = xt
                states[li], xt = blk.step(states[li], xt)
                if return_traj and blk.arm in NEEDS_TRAJ:
                    trajs[li].append(states[li]); xseqs[li].append(h_in if False else blk.ln1(h_in))
            outs.append(self.readout(self.ln_f(xt)))
        logits = torch.stack(outs, 1)
        if not return_traj:
            return logits
        traj_out = {li: (torch.stack(trajs[li], 1), torch.stack(xseqs[li], 1))
                    for li in range(len(self.blocks)) if trajs[li]}
        return logits, traj_out

## 3. Lyapunov estimator (unchanged mechanism, now per-layer)

Applied independently to each block whose arm needs it (`chaossm_free`, `chaossm_corridor`),
using that block's own `cell.step` and its own recorded trajectory/input sequence. Penalties
from multiple such layers are summed.

In [ ]:
from torch.func import jacrev

def _full_jac_fn(cell, x):
    return lambda z: cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(cell, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(cell, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 4. Task, chance, held-out evaluation set (fixed across all arms/seeds/checkpoints)

In [ ]:
def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

VOCAB_SIZE = CFG.vocab + 2
CHANCE = 1.0 / CFG.vocab

def copy_accuracy(logits, tgt):
    mask = tgt != -100
    pred = logits.argmax(-1)
    return ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()

# Fixed held-out set: generated ONCE with a dedicated seed, identical for every arm/seed/run.
_eval_gen = torch.Generator(device=DEVICE if DEVICE.type == "cuda" else "cpu")
_eval_gen.manual_seed(CFG.eval_seed)
EVAL_INP, EVAL_TGT = make_copy_batch(CFG, CFG.eval_n, DEVICE, _eval_gen)
print(f"held-out eval set: {CFG.eval_n} sequences, seed={CFG.eval_seed}, shape={tuple(EVAL_INP.shape)}")

@torch.no_grad()
def eval_held_out(model, cfg, chunk=256):
    model.eval()
    correct, total = 0.0, 0.0
    for i in range(0, cfg.eval_n, chunk):
        inp = EVAL_INP[i:i+chunk]; tgt = EVAL_TGT[i:i+chunk]
        logits = model(inp)
        mask = tgt != -100
        pred = logits.argmax(-1)
        correct += ((pred == tgt) & mask).float().sum().item()
        total += mask.float().sum().item()
    model.train()
    return correct / max(total, 1)

held-out eval set: 2000 sequences, seed=12345, shape=(2000, 25)


## 5. Cosine schedule with warmup

In [ ]:
def make_lr_lambda(cfg, total_steps):
    warmup = max(1, int(cfg.warmup_frac * total_steps))
    def fn(step):
        if step < warmup:
            return step / warmup
        prog = (step - warmup) / max(1, total_steps - warmup)
        cos = 0.5 * (1 + math.cos(math.pi * prog))
        return cfg.lr_min_frac + (1 - cfg.lr_min_frac) * cos
    return fn

## 6. Trainer

In [ ]:
def train_one(arm, seed, cfg, lr, total_steps, track_eval_curve=True):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type == "cuda" else "cpu"); gen.manual_seed(seed)
    model = ChaoSSMLM(cfg, arm, VOCAB_SIZE, VOCAB_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, make_lr_lambda(cfg, total_steps))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = max(1, int(cfg.warmup_frac * total_steps))
    diverged = False
    eval_hist = []

    for step in range(total_steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = arm in NEEDS_TRAJ
        out = model(inp, return_traj=need_traj)
        logits, traj_out = out if need_traj else (out, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE)
        lam_val = float("nan")
        if need_traj and traj_out:
            in_warmup = step < warmup_steps
            lams = []
            for li, (traj, xseq) in traj_out.items():
                lam = lambda_window(model.blocks[li].cell, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
                lams.append(lam)
                if use_lyap and not in_warmup:
                    L_pen = L_pen + a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)
            if lams:
                lam_val = float(torch.stack(lams).mean().detach())

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step(); sched.step()

        if use_lyap and step >= warmup_steps and math.isfinite(lam_val):
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if track_eval_curve and (step % cfg.eval_every == 0 or step == total_steps - 1):
            acc = eval_held_out(model, cfg)
            eval_hist.append(dict(step=step, arm=arm, seed=seed, lr=lr, held_out_acc=acc,
                                   lam=lam_val, gnorm=gn))

    final_acc = eval_held_out(model, cfg) if not diverged else float("nan")
    return (dict(arm=arm, seed=seed, lr=lr, steps=total_steps, diverged=diverged,
                 final_held_out_acc=final_acc),
            pd.DataFrame(eval_hist))

## 7. Main run -- default lr per arm, full budget, 8 seeds

In [ ]:
rows, hists = [], []
total = len(CFG.arms) * CFG.seeds
done = 0
for arm in CFG.arms:
    lr = CFG.default_lr[arm]
    for sd in range(CFG.seeds):
        s, h = train_one(arm, sd, CFG, lr, CFG.steps)
        rows.append(s); hists.append(h); done += 1
        flag = " DIVERGED" if s["diverged"] else ""
        print(f"[{done}/{total}] {arm:18s} seed={sd} lr={lr:.0e} "
              f"held_out_acc={s['final_held_out_acc']:.4f} (chance {CHANCE:.4f}){flag}")

MAIN = pd.DataFrame(rows); MAIN_HIST = pd.concat(hists, ignore_index=True)
savetable(MAIN, "phase3b_prime_main_summary.csv")
savetable(MAIN_HIST, "phase3b_prime_main_history.csv")
print()
print(MAIN.groupby("arm").final_held_out_acc.agg(["mean","std"]).round(4).to_string())

[1/40] linear             seed=0 lr=3e-04 held_out_acc=0.1329 (chance 0.0625)
[2/40] linear             seed=1 lr=3e-04 held_out_acc=0.0698 (chance 0.0625)
[3/40] linear             seed=2 lr=3e-04 held_out_acc=0.1216 (chance 0.0625)
[4/40] linear             seed=3 lr=3e-04 held_out_acc=0.1280 (chance 0.0625)
[5/40] linear             seed=4 lr=3e-04 held_out_acc=0.0625 (chance 0.0625)
[6/40] linear             seed=5 lr=3e-04 held_out_acc=0.0619 (chance 0.0625)
[7/40] linear             seed=6 lr=3e-04 held_out_acc=0.1897 (chance 0.0625)
[8/40] linear             seed=7 lr=3e-04 held_out_acc=0.1645 (chance 0.0625)
[9/40] gru                seed=0 lr=1e-03 held_out_acc=0.8518 (chance 0.0625)
[10/40] gru                seed=1 lr=1e-03 held_out_acc=0.8333 (chance 0.0625)
[11/40] gru                seed=2 lr=1e-03 held_out_acc=0.9118 (chance 0.0625)
[12/40] gru                seed=3 lr=1e-03 held_out_acc=0.8275 (chance 0.0625)
[13/40] gru                seed=4 lr=1e-03 held_out_acc=0.806

## 8. Automatic fallback sweep for any arm below the ceiling bar

Per H-A / the plan's "3-point sweep": any arm whose mean held-out accuracy is below
`ceiling_bar` (0.9) gets a short, half-budget, 3-seed sweep over `sweep_grid` (excluding its
already-tried default lr), then a full 8-seed / full-budget rerun at whichever lr scored
highest on held-out accuracy in the sweep. This replaces that arm's row in `MAIN` --
`FINAL` below is the authoritative table.

In [10]:
FINAL_rows, FINAL_hists = list(rows), list(hists)
sweep_log = []
means = MAIN.groupby("arm").final_held_out_acc.mean()
sweep_steps = max(1, int(CFG.sweep_step_frac * CFG.steps))

for arm in CFG.arms:
    if means.get(arm, 0.0) >= CFG.ceiling_bar:
        continue
    print(f"\n--- {arm}: mean {means[arm]:.4f} < bar {CFG.ceiling_bar}. Running fallback lr sweep ---")
    grid = [lr for lr in CFG.sweep_grid if not math.isclose(lr, CFG.default_lr[arm], rel_tol=1e-6)]
    sweep_scores = {}
    for lr in grid:
        accs = []
        for sd in range(CFG.sweep_seeds):
            s, _ = train_one(arm, sd, CFG, lr, sweep_steps, track_eval_curve=False)
            accs.append(s["final_held_out_acc"])
        m = float(np.nanmean(accs))
        sweep_scores[lr] = m
        sweep_log.append(dict(arm=arm, lr=lr, sweep_steps=sweep_steps,
                               sweep_seeds=CFG.sweep_seeds, mean_held_out_acc=m))
        print(f"    lr={lr:.0e}  sweep mean held_out_acc={m:.4f}")
    best_lr = max(sweep_scores, key=sweep_scores.get)
    print(f"  selected lr={best_lr:.0e} for {arm}'s final full-budget run")

    # drop this arm's rows from FINAL and rerun at full budget/seeds with best_lr
    FINAL_rows = [r for r in FINAL_rows if r["arm"] != arm]
    FINAL_hists = [h for h in FINAL_hists if not (len(h) and h["arm"].iloc[0] == arm)]
    for sd in range(CFG.seeds):
        s, h = train_one(arm, sd, CFG, best_lr, CFG.steps)
        FINAL_rows.append(s); FINAL_hists.append(h)
        print(f"    [final] {arm} seed={sd} lr={best_lr:.0e} "
              f"held_out_acc={s['final_held_out_acc']:.4f}"
              f"{' DIVERGED' if s['diverged'] else ''}")

SWEEP = pd.DataFrame(sweep_log)
if len(SWEEP): savetable(SWEEP, "phase3b_prime_sweep_log.csv")
FINAL = pd.DataFrame(FINAL_rows); FINAL_HIST = pd.concat(FINAL_hists, ignore_index=True)
savetable(FINAL, "phase3b_prime_final_summary.csv")
savetable(FINAL_HIST, "phase3b_prime_final_history.csv")
print()
print(FINAL.groupby("arm").final_held_out_acc.agg(["mean","std"]).round(4).to_string())


--- linear: mean 0.1164 < bar 0.9. Running fallback lr sweep ---
    lr=1e-04  sweep mean held_out_acc=0.0873
    lr=1e-03  sweep mean held_out_acc=0.0648
    lr=3e-03  sweep mean held_out_acc=0.0632
  selected lr=1e-04 for linear's final full-budget run
    [final] linear seed=0 lr=1e-04 held_out_acc=0.1308
    [final] linear seed=1 lr=1e-04 held_out_acc=0.1055
    [final] linear seed=2 lr=1e-04 held_out_acc=0.0971
    [final] linear seed=3 lr=1e-04 held_out_acc=0.1352
    [final] linear seed=4 lr=1e-04 held_out_acc=0.0605
    [final] linear seed=5 lr=1e-04 held_out_acc=0.0912
    [final] linear seed=6 lr=1e-04 held_out_acc=0.1177
    [final] linear seed=7 lr=1e-04 held_out_acc=0.0986

--- gru: mean 0.8311 < bar 0.9. Running fallback lr sweep ---
    lr=1e-04  sweep mean held_out_acc=0.2265
    lr=3e-04  sweep mean held_out_acc=0.3093
    lr=3e-03  sweep mean held_out_acc=0.9236
  selected lr=3e-03 for gru's final full-budget run
    [final] gru seed=0 lr=3e-03 held_out_acc=0.9958
  

## 9. Statistics -- Cliff's delta, bootstrap CI, Holm-Bonferroni correction

Replaces the floor-p-value-only reporting used through Phase 3a. All pairwise arm
comparisons on this task are corrected jointly (Holm), not read individually.

In [11]:
def cliffs_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    gt = sum((xi > y).sum() for xi in x)
    lt = sum((xi < y).sum() for xi in x)
    return (gt - lt) / (len(x) * len(y))

def bootstrap_ci_diff(x, y, n_boot=10000, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(x); y = np.asarray(y)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        bx = rng.choice(x, size=len(x), replace=True)
        by = rng.choice(y, size=len(y), replace=True)
        diffs[i] = bx.mean() - by.mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(lo), float(hi)

def holm_correction(pvals):
    idx = np.argsort(pvals)
    m = len(pvals)
    adj = np.empty(m)
    running_max = 0.0
    for rank, i in enumerate(idx):
        val = (m - rank) * pvals[i]
        running_max = max(running_max, val)
        adj[i] = min(running_max, 1.0)
    return adj

from itertools import combinations
pairs = list(combinations(CFG.arms, 2))
stat_rows = []
for a, b_ in pairs:
    x = FINAL[FINAL.arm == a].sort_values("seed").final_held_out_acc.values
    y = FINAL[FINAL.arm == b_].sort_values("seed").final_held_out_acc.values
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    n = min(len(x), len(y))
    if n < 3 or np.allclose(x[:n], y[:n]):
        p = np.nan
    else:
        try:
            _, p = wilcoxon(x[:n], y[:n])
        except ValueError:
            p = np.nan
    delta = cliffs_delta(x, y) if len(x) and len(y) else np.nan
    ci_lo, ci_hi = bootstrap_ci_diff(x, y) if len(x) and len(y) else (np.nan, np.nan)
    stat_rows.append(dict(pair=f"{a} vs {b_}", n=n, p_raw=p,
                           cliffs_delta=delta, mean_diff=float(np.mean(x)-np.mean(y)) if len(x) and len(y) else np.nan,
                           ci_low=ci_lo, ci_high=ci_hi))

STATS = pd.DataFrame(stat_rows)
valid = STATS["p_raw"].notna()
STATS["p_holm"] = np.nan
STATS.loc[valid, "p_holm"] = holm_correction(STATS.loc[valid, "p_raw"].values)
print(STATS.round(4).to_string(index=False))
savetable(STATS, "phase3b_prime_stats.csv")

                            pair  n  p_raw  cliffs_delta  mean_diff  ci_low  ci_high  p_holm
                   linear vs gru  8 0.0078       -1.0000    -0.8901 -0.9067  -0.8754  0.0781
                linear vs mamba2  8 0.0078       -1.0000    -0.2231 -0.2657  -0.1804  0.0781
          linear vs chaossm_free  8 0.0078       -1.0000    -0.0429 -0.0598  -0.0278  0.0781
      linear vs chaossm_corridor  8 0.0078       -1.0000    -0.0457 -0.0637  -0.0289  0.0781
                   gru vs mamba2  8 0.0078        1.0000     0.6670  0.6279   0.7067  0.0781
             gru vs chaossm_free  8 0.0078        1.0000     0.8471  0.8427   0.8511  0.0781
         gru vs chaossm_corridor  8 0.0078        1.0000     0.8444  0.8359   0.8514  0.0781
          mamba2 vs chaossm_free  8 0.0078        1.0000     0.1801  0.1399   0.2200  0.0781
      mamba2 vs chaossm_corridor  8 0.0078        1.0000     0.1774  0.1364   0.2178  0.0781
chaossm_free vs chaossm_corridor  8 0.6406       -0.0312    -0.0028 -0

## 10. Figures

In [12]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
order = list(CFG.arms)
means_f = FINAL.groupby("arm").final_held_out_acc.mean().reindex(order)
stds_f = FINAL.groupby("arm").final_held_out_acc.std().reindex(order)
colors = {"linear":"gray","gru":"tab:purple","mamba2":"tab:orange",
          "chaossm_free":"tab:blue","chaossm_corridor":"tab:green"}
ax.bar(range(len(order)), means_f.values, yerr=stds_f.values, capsize=4,
       color=[colors[a] for a in order])
ax.axhline(CHANCE, color="r", ls="--", label="chance")
ax.axhline(CFG.ceiling_bar, color="k", ls=":", label=f"ceiling bar ({CFG.ceiling_bar})")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=15, ha="right")
ax.set(ylabel="held-out copy accuracy", ylim=(0,1.02),
       title="Phase 3b': held-out accuracy after harness fix (final lr used per arm)")
ax.legend(); ax.grid(alpha=.3, axis="y")
fig.tight_layout(); savefig(fig, "phase3b_prime_final_bars.png"); plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for arm in order:
    sub = FINAL_HIST[FINAL_HIST.arm == arm]
    curve = sub.groupby("step").held_out_acc.mean()
    ax.plot(curve.index, curve.values, label=arm, color=colors[arm])
ax.axhline(CHANCE, color="r", ls="--", lw=1, label="chance")
ax.axhline(CFG.ceiling_bar, color="k", ls=":", lw=1, label="ceiling bar")
ax.set(xlabel="training step", ylabel="held-out copy accuracy", ylim=(0,1.02),
       title="Phase 3b': held-out accuracy over training (mean over seeds)")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase3b_prime_learning_curves.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase3b_prime/phase3b_prime_final_bars.png
saved /content/drive/My Drive/ChaoSSM/results/phase3b_prime/phase3b_prime_learning_curves.png


## 11. Verdict (H-A)

In [13]:
bar = CFG.ceiling_bar
final_means = FINAL.groupby("arm").final_held_out_acc.mean().reindex(list(CFG.arms))
gru_pass = bool(final_means.get("gru", 0.0) >= bar)
chaotic_pass = {a: bool(final_means.get(a, 0.0) >= bar) for a in ("chaossm_free", "chaossm_corridor")}
any_chaotic_pass = any(chaotic_pass.values())
mamba2_pass = bool(final_means.get("mamba2", 0.0) >= bar)
linear_pass = bool(final_means.get("linear", 0.0) >= bar)

if not gru_pass:
    decision = ("H-A INCONCLUSIVE / HARNESS STILL BROKEN: GRU did not reach the ceiling bar "
                "even after the fallback sweep. Do not draw architecture conclusions from this "
                "harness yet -- investigate task encoding, eval set validity, or optimizer "
                "settings before proceeding to Phase 4.")
elif gru_pass and any_chaotic_pass:
    decision = ("H-A CONFIRMED: GRU and at least one chaotic arm both reach the ceiling bar. "
                "The trainability question is closed. Proceed to Phase 4 (causal isolation of "
                "chaos via the periodic-r ablation) on this harness.")
else:
    decision = ("H-A: GRU reaches the ceiling bar but no chaotic arm does. Reported as a "
                "limitation: chaos (as currently regularized/parametrized) impairs trainability "
                "on copy at this scale relative to a standard gated RNN. Phase 4's periodic-r "
                "ablation still runs -- it answers a different question (is whatever accuracy "
                "chaos does reach attributable to chaos or to nonlinearity alone) -- but the "
                "manuscript framing shifts toward a negative/characterization result.")

verdict = dict(
    smoke_test=SMOKE_TEST,
    ceiling_bar=bar,
    final_means=final_means.round(4).to_dict(),
    gru_pass=gru_pass, mamba2_pass=mamba2_pass, linear_pass=linear_pass,
    chaotic_pass=chaotic_pass, any_chaotic_pass=any_chaotic_pass,
    lr_used=dict(list(CFG.default_lr.items())),
    sweep_triggered_for=sorted(set(sweep_log_arm for sweep_log_arm in
                                    (SWEEP["arm"].unique().tolist() if len(SWEEP) else []))),
    any_diverged=bool(FINAL.diverged.any()),
    stats_summary=STATS[["pair","n","p_raw","p_holm","cliffs_delta","mean_diff"]].round(4).to_dict("records"),
    decision=decision,
    config=dict(steps=CFG.steps, seeds=CFG.seeds, arms=CFG.arms, n_layers=CFG.n_layers,
                eval_n=CFG.eval_n, eval_seed=CFG.eval_seed),
)
savejson(verdict, "phase3b_prime_verdict.json")
print(decision)
print()
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase3b_prime/phase3b_prime_verdict.json
H-A: GRU reaches the ceiling bar but no chaotic arm does. Reported as a limitation: chaos (as currently regularized/parametrized) impairs trainability on copy at this scale relative to a standard gated RNN. Phase 4's periodic-r ablation still runs -- it answers a different question (is whatever accuracy chaos does reach attributable to chaos or to nonlinearity alone) -- but the manuscript framing shifts toward a negative/characterization result.

{
  "smoke_test": false,
  "ceiling_bar": 0.9,
  "final_means": {
    "linear": 0.1046,
    "gru": 0.9946,
    "mamba2": 0.3276,
    "chaossm_free": 0.1475,
    "chaossm_corridor": 0.1503
  },
  "gru_pass": true,
  "mamba2_pass": false,
  "linear_pass": false,
  "chaotic_pass": {
    "chaossm_free": false,
    "chaossm_corridor": false
  },
  "any_chaotic_pass": false,
  "lr_used": {
    "linear": 0.0003,
    "chaossm_free": 0.0003,
    "chaossm_corridor": 0